# Durable Agent Runtime and MCP

Build a local review workflow with typed tool access, approvals, durable state, and replay-safe execution.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Connect and constrain


### Lecture 01 · Discover and call a local MCP tool

Trace the MCP initialization, tool discovery, and tool invocation over stdio.

Run the discovery cell. Call lookup_agenda for the sample meeting and inspect the JSON-RPC result.


In [ ]:
import json
import subprocess
import sys

# Minimal local MCP server fixture. Each line of stdout is a protocol message.
server = """
import json
import sys

for line in sys.stdin:
    message = json.loads(line)
    method = message["method"]
    if method == "notifications/initialized":
        continue
    reply = {"jsonrpc": "2.0", "id": message["id"]}
    if method == "initialize":
        reply["result"] = {
            "protocolVersion": "2025-06-18",
            "capabilities": {"tools": {}},
            "serverInfo": {"name": "agenda-fixture", "version": "1.0"},
        }
    elif method == "tools/list":
        reply["result"] = {"tools": [{
            "name": "lookup_agenda",
            "description": "Read the local synthetic review agenda.",
            "inputSchema": {
                "type": "object", "properties": {"meeting": {"type": "string", "enum": ["M1"]}},
                "required": ["meeting"], "additionalProperties": False,
            },
        }]}
    elif method == "tools/call":
        params = message["params"]
        if params.get("name") == "lookup_agenda" and params.get("arguments") == {"meeting": "M1"}:
            reply["result"] = {"content": [{"type": "text", "text": "Review the accessibility checklist."}]}
        else:
            reply["error"] = {"code": -32602, "message": "Invalid tool or arguments"}
    else:
        reply["error"] = {"code": -32601, "message": "Method not found"}
    print(json.dumps(reply), flush=True)
"""


def exchange(method, params=None):
    messages = [
        {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
            "protocolVersion": "2025-06-18", "capabilities": {},
            "clientInfo": {"name": "aiebok-workbook", "version": "1.0"},
        }},
        {"jsonrpc": "2.0", "method": "notifications/initialized"},
        {"jsonrpc": "2.0", "id": 2, "method": method, "params": params or {}},
    ]
    run = subprocess.run(
        [sys.executable, "-c", server], input="".join(json.dumps(msg) + "\n" for msg in messages),
        text=True, capture_output=True, timeout=5, check=True,
    )
    init, reply = [json.loads(line) for line in run.stdout.splitlines()]
    assert init["id"] == 1 and init["result"]["protocolVersion"] == "2025-06-18"
    assert reply["id"] == 2 and reply["jsonrpc"] == "2.0"
    return reply["result"]


tools = exchange("tools/list")["tools"]
assert [tool["name"] for tool in tools] == ["lookup_agenda"]
assert tools[0]["inputSchema"]["required"] == ["meeting"]
print("Discovered", tools)


**Compare your result**

The server returns one typed tool declaration and a structured read result over the local process boundary.


### Lecture 02 · Tool authorization and argument validation

Validate caller scope and arguments before dispatching a tool.

Run the allowed, denied, and invalid argument cases. Inspect where each decision is recorded.


In [ ]:
events = []


def lookup(role, meeting):
    if role != "reviewer":
        events.append("denied")
        return None
    if meeting != "M1":
        events.append("invalid_arguments")
        return None
    result = exchange("tools/call", {"name": "lookup_agenda", "arguments": {"meeting": meeting}})
    events.append("read")
    return result["content"][0]["text"]


assert lookup("guest", "M1") is None
assert lookup("reviewer", "M9") is None
assert lookup("reviewer", "M1") == "Review the accessibility checklist."
assert events == ["denied", "invalid_arguments", "read"]


**Compare your result**

Only a caller with the approved role and valid meeting ID reaches the tool.


## Pause, recover and replay


### Lecture 03 · Persistent approval and checkpoint recovery

Persist a proposal and its approval against a payload fingerprint, then reopen the runtime.

Reopen the SQLite store after creating a proposal. Verify that pending work stays pending and compare a mismatched approval.


In [ ]:
from hashlib import sha256
from pathlib import Path
import sqlite3
from tempfile import TemporaryDirectory


def open_store(path):
    db = sqlite3.connect(path)
    db.executescript("""
        CREATE TABLE IF NOT EXISTS proposals (
          id TEXT PRIMARY KEY, payload TEXT NOT NULL, digest TEXT NOT NULL, approved INTEGER NOT NULL
        );
        CREATE TABLE IF NOT EXISTS notes (id TEXT PRIMARY KEY, payload TEXT NOT NULL);
    """)
    return db


def propose(db, identifier, payload):
    digest = sha256(payload.encode()).hexdigest()
    with db:
        db.execute("INSERT INTO proposals VALUES (?, ?, ?, 0)", (identifier, payload, digest))
    return digest


def approve(db, identifier, digest):
    with db:
        return db.execute("UPDATE proposals SET approved = 1 WHERE id = ? AND digest = ?",
                          (identifier, digest)).rowcount == 1


directory = TemporaryDirectory(prefix="aiebok-review-")
database = Path(directory.name) / "review.sqlite"
db = open_store(database)
digest = propose(db, "proposal-1", "M1 has completed accessibility review")
db.close()
db = open_store(database)
assert db.execute("SELECT approved FROM proposals").fetchone() == (0,)
assert not approve(db, "proposal-1", "different-payload")
assert db.execute("SELECT approved FROM proposals").fetchone() == (0,)


**Compare your result**

The record survives a restart and the wrong fingerprint leaves it awaiting approval.


### Lecture 04 · Idempotent execution and decision traces

Commit a local effect exactly once across retries and distinguish external effects.

Reopen the store again and replay the approved action. Check the persistent effect count and trace.


In [ ]:
def execute(db, identifier):
    with db:
        db.execute("BEGIN IMMEDIATE")
        row = db.execute("SELECT payload, approved FROM proposals WHERE id = ?", (identifier,)).fetchone()
        if row is None or not row[1]:
            return "awaiting_approval"
        effect = db.execute("INSERT OR IGNORE INTO notes VALUES (?, ?)", (identifier, row[0]))
        return "completed" if effect.rowcount else "replayed"


assert execute(db, "proposal-1") == "awaiting_approval"
assert approve(db, "proposal-1", digest)
assert execute(db, "proposal-1") == "completed"
db.close()
db = open_store(database)
try:
    assert execute(db, "proposal-1") == "replayed"
    assert db.execute("SELECT count(*) FROM notes").fetchone() == (1,)
    print("Decision trace", events + ["awaiting_approval", "approved", "completed", "replayed"])
finally:
    db.close()
    directory.cleanup()


**Compare your result**

The replay reports the existing result and there is one note. Explain why this guarantee would change for a remote calendar API.


## Course project

Build a supervised agenda-review workflow using an MCP-compatible local tool. Show validated calls, approval, a process restart, and no duplicate effect.

Capture a tool discovery transcript, a denied tool call, a human approval, and a replay from reopened state. Document which guarantees depend on the local transaction.
